# 🧥 ARVTON — AR/VR Virtual Try-On · Full Training Notebook

> **Run this notebook top-to-bottom on Google Colab (T4 free tier or A100 Pro)**
## What this notebook does
| Step | Description | Est. Time |
|------|-------------|----------|
| 1 | GPU check + environment setup | ~3 min |
| 2 | Clone repo + install dependencies | ~4 min |
| 3 | Mount Google Drive | 1 min |
| 4 | Prepare dataset (synthetic or custom) | ~5-15 min |
| 5 | Configure training hyperparameters | instant |
| 6 | **Train GAN model** | ~240 min (4 hrs) |
| 7 | Plot loss curves + evaluate | ~2 min |
| 8 | Save checkpoints to Drive + download | ~1 min |
---
### Architecture Overview
```
Person Photo + Garment Image
        │
     [SAM 2] → Segmentation masks
  [Leffa / CatVTON] → 2D try-on composite
  [HMR 2.0 + ECON] → SMPL mesh
  [TripoSR / SyncHuman] → .glb 3D model
  [RefineGenerator GAN] ← Fine-tuned here
**License:** MIT — Only CC0 / Apache 2.0 / MIT data & weights used.

---
## Step 1 — Check GPU & Python environment

In [ ]:
# ─── GPU & CUDA check ────────────────────────────────────────────────
!nvidia-smi
import sys, torch

print(f"\nPython  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"CUDA OK : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1024**3
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {vram:.1f} GB")
    if vram < 12:
        print("⚠️  Low VRAM — use BATCH_SIZE=1 and IMAGE_SIZE=(384,512)")
    elif vram < 20:
        print("✅ T4 detected — recommended settings pre-loaded below")
    else:
        print("🚀 A100/H100 detected — can use larger batch & resolution")
else:
    print("❌ No GPU found — go to Runtime → Change runtime type → T4 GPU")

---
## Step 2 — Clone repository & install dependencies

In [ ]:
import os

REPO_URL  = "https://github.com/Gandharv2323/AR-PROTOTYPE-4.git"
REPO_DIR  = "/content/AR-PROTOTYPE-4"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print("✅ Repo cloned")
else:
    !cd {REPO_DIR} && git pull --quiet
    print("✅ Repo up-to-date")
%cd {REPO_DIR}
!ls -la

In [ ]:
# Install all ARVTON dependencies (skip heavy 3D libs for training-only)
!pip install -q \
    torch torchvision --index-url https://download.pytorch.org/whl/cu121 \
    && pip install -q \
    peft>=0.10.0 \
    accelerate>=0.29.0 \
    diffusers>=0.27.0 \
    transformers>=4.40.0 \
    open-clip-torch>=2.24.0 \
    opencv-python-headless>=4.9.0 \
    Pillow>=10.3.0 \
    scikit-image>=0.22.0 \
    numpy scipy tqdm matplotlib \
    fastapi uvicorn python-multipart

print("✅ Core dependencies installed")

In [ ]:
# Run the project setup script (creates dirs, verifies imports)
!python setup_local.py
print("✅ Project directories created")

---
## Step 3 — Mount Google Drive

All checkpoints will be saved to `MyDrive/ARVTON/checkpoints/` so they **persist after the Colab session ends**.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_CKPT  = Path("/content/drive/MyDrive/ARVTON/checkpoints")
DRIVE_DATA  = Path("/content/drive/MyDrive/ARVTON/datasets")
DRIVE_LOGS  = Path("/content/drive/MyDrive/ARVTON/logs")
for d in [DRIVE_CKPT, DRIVE_DATA, DRIVE_LOGS]:
    d.mkdir(parents=True, exist_ok=True)
print(f"✅ Checkpoints → {DRIVE_CKPT}")
print(f"✅ Datasets    → {DRIVE_DATA}")
print(f"✅ Logs        → {DRIVE_LOGS}")

---
## Step 4 — Prepare Dataset

Choose **one** of the three options:

In [ ]:
# ── CHOOSE YOUR DATA SOURCE ───────────────────────────────────────────
#   'synthetic'  → generate from Stable Diffusion (no data needed)
#   'upload'     → upload your own zip file of image pairs
#   'drive'      → load from a zip already on Google Drive

DATA_SOURCE   = 'synthetic'   # ← change this
NUM_SYNTHETIC = 150           # number of synthetic pairs to generate
DRIVE_ZIP     = "/content/drive/MyDrive/ARVTON/datasets/my_dataset.zip"  # for 'drive' option

In [ ]:
import json, shutil
from pathlib import Path

MANIFEST_PATH = Path("data/arvton/datasets/train_manifest.json")
if DATA_SOURCE == 'synthetic':
    print(f"Generating {NUM_SYNTHETIC} synthetic training pairs...")
    print(f"Estimated time: ~{NUM_SYNTHETIC * 3 // 60 + 1} min")
    !python -m pipeline.synthetic_gen --target-count {NUM_SYNTHETIC}
    print("✅ Synthetic dataset generated")
elif DATA_SOURCE == 'upload':
    from google.colab import files
    print("Upload your dataset.zip (containing person_*.jpg, garment_*.jpg, mask_*.png)")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !unzip -q {zip_name} -d data/arvton/datasets/custom/
    print("✅ Dataset uploaded and extracted")
elif DATA_SOURCE == 'drive':
    zip_src = Path(DRIVE_ZIP)
    if zip_src.exists():
        !cp "{zip_src}" /content/dataset.zip
        !unzip -q /content/dataset.zip -d data/arvton/datasets/custom/
        print("✅ Dataset loaded from Drive")
    else:
        print(f"❌ File not found: {zip_src}")

In [ ]:
# ── Verify dataset ────────────────────────────────────────────────────
import json
from pathlib import Path

manifest = Path("data/arvton/datasets/train_manifest.json")
if manifest.exists():
    raw = json.loads(manifest.read_text())
    records = raw if isinstance(raw, list) else raw.get("entries", [])
    print(f"✅ Manifest found: {len(records)} training pairs")
    if records:
        print("  Sample keys:", list(records[0].keys()))
else:
    print("⚠️  No manifest found — run a data option above first")
# Count image files
data_root = Path("data/arvton/datasets")
persons   = list(data_root.rglob("person_*.jpg")) + list(data_root.rglob("person_*.png"))
garments  = list(data_root.rglob("garment_*.jpg")) + list(data_root.rglob("garment_*.png"))
masks     = list(data_root.rglob("mask_*.png"))
print(f"  Person images  : {len(persons)}")
print(f"  Garment images : {len(garments)}")
print(f"  Mask images    : {len(masks)}")

---
## Step 5 — Configure Training Hyperparameters

In [ ]:
import torch

# ── Detect GPU tier ───────────────────────────────────────────────────
if torch.cuda.is_available():
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
else:
    vram_gb  = 0
    gpu_name = "CPU"
# ── Auto-tune defaults based on VRAM ──────────────────────────────────
if vram_gb >= 35:          # A100 40 GB / H100
    BATCH_SIZE = 8
    IMAGE_SIZE = (768, 1024)
    LORA_RANK  = 32
    PRECISION  = "bf16"
elif vram_gb >= 15:        # T4 16 GB (free tier)
    BATCH_SIZE = 2         # 3 can OOM with LoRA+AMP at 576x576; keep at 2
    IMAGE_SIZE = (512, 512)
    LORA_RANK  = 16
    PRECISION  = "fp16"
else:                      # 8 GB / CPU fallback
    BATCH_SIZE = 1
    IMAGE_SIZE = (384, 512)
    LORA_RANK  = 8
# ── Override any value here ────────────────────────────────────────────
EPOCHS           = 180     # Total training epochs
LR_GENERATOR     = 2e-4   # Generator LR  (AdamW)
LR_DISCRIMINATOR = 2e-4   # Discriminator LR
USE_AMP          = True    # Mixed precision (FP16/BF16) — saves VRAM
USE_LORA         = True    # LoRA adapters — reduces trainable params to <5 %
SAVE_EVERY       = 20      # Checkpoint every N epochs (+ always on best val)
NUM_WORKERS      = 2       # DataLoader worker processes
VAL_SPLIT        = 0.08    # 8 % of data held out for validation
GRAD_ACCUM_STEPS = 4       # Effective batch = BATCH_SIZE × GRAD_ACCUM_STEPS
EMA_DECAY         = 0.999      # EMA decay rate (0.999 = slow, stable)
EARLY_STOP_PATIENCE = 20       # Stop if no improvement for N epochs
EARLY_STOP_MIN_DELTA = 1e-4   # Min improvement to count as progress
WARMUP_EPOCHS     = 5          # LR warmup epochs
USE_SPECTRAL_NORM = True      # Spectral norm for discriminator
USE_WANDB         = False     # Weights & Biases logging
GRAD_CLIP_NORM   = 1.0     # Max L2 norm for gradient clipping (GAN stability)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
est_min = EPOCHS * 1.2
print("=" * 58)
print(f"  GPU            : {gpu_name}")
print(f"  VRAM           : {vram_gb:.1f} GB")
print(f"  Device         : {DEVICE}")
print("-" * 58)
print(f"  Epochs         : {EPOCHS}")
print(f"  Batch size     : {BATCH_SIZE}  (effective: {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"  Image size     : {IMAGE_SIZE}")
print(f"  LR (G / D)     : {LR_GENERATOR} / {LR_DISCRIMINATOR}")
print(f"  Mixed prec.    : {PRECISION if USE_AMP else 'disabled'}")
print(f"  LoRA           : {'ON  (rank=' + str(LORA_RANK) + ')' if USE_LORA else 'OFF'}")
print(f"  Grad accum.    : {GRAD_ACCUM_STEPS} steps")
print(f"  Grad clip norm : {GRAD_CLIP_NORM}")
print(f"  Val split      : {VAL_SPLIT * 100:.0f} %")
print(f"  Est. time (T4) : ~{est_min:.0f} min  (~{est_min / 60:.1f} hrs)")

---
## Step 6 — Train the ARVTON GAN Model

This trains the `RefineGenerator` (U-Net) + `MultiScaleDiscriminator` using:
- **L1 reconstruction loss** + **VGG perceptual loss** + **GAN adversarial loss**
- **LoRA adapters** (optional, reduces trainable params to < 5%)
- **Mixed precision** (FP16/BF16) + **gradient checkpointing**

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────
import json, logging, time, shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from pipeline.fine_tune   import TryOnDataset, apply_lora
from pipeline.refine      import RefineGenerator, MultiScaleDiscriminator, TryOnCombinedLoss
from pipeline.platform_utils import setup_logging, get_paths
setup_logging("INFO")
logger = logging.getLogger("arvton.colab")
paths  = get_paths()
print("✅ Imports OK")

In [ ]:
# ── Build datasets & loaders ──────────────────────────────────────────
MANIFEST = str(paths["train_manifest"])

full_ds = TryOnDataset(
    manifest_path=MANIFEST,
    image_size=IMAGE_SIZE,
    augment=True,
)
val_size   = max(1, int(len(full_ds) * VAL_SPLIT))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])
train_loader = DataLoader(
# Use optimized dataloader
train_loader = create_optimized_dataloader(
    train_ds, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2,
)
val_loader = create_optimized_dataloader(
    val_ds, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2,
)
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
print(f"✅ Dataset ready")
print(f"   Train: {train_size} samples ({len(train_loader)} batches/epoch)")
print(f"   Val  : {val_size} samples")

In [ ]:
# ── Build Generator + Discriminator ───────────────────────────────────
generator     = RefineGenerator(in_channels=6, out_channels=3).to(DEVICE)
discriminator = MultiScaleDiscriminator(in_channels=3).to(DEVICE)

if USE_LORA:
    generator = apply_lora(generator, rank=LORA_RANK, alpha=LORA_RANK * 2)
# Trainable parameter counts
g_params = sum(p.numel() for p in generator.parameters() if p.requires_grad)
d_params = sum(p.numel() for p in discriminator.parameters() if p.requires_grad)
total    = g_params + d_params
print(f"✅ Models built")
print(f"   Generator params     : {g_params:,}")
print(f"   Discriminator params : {d_params:,}")
print(f"   Total trainable      : {total:,}")
# Enable gradient checkpointing on generator
if hasattr(generator, 'gradient_checkpointing_enable'):
    generator.gradient_checkpointing_enable()
    print("   Gradient checkpointing: ON")
# Initialize Early Stopping
early_stopping = EarlyStopping(
    patience=EARLY_STOP_PATIENCE,
    min_delta=EARLY_STOP_MIN_DELTA,
    mode="min",
)

# Initialize EMA for generator
ema_generator = ExponentialMovingAverage(generator, decay=EMA_DECAY, device=DEVICE)
print(f"   Early stopping : patience={EARLY_STOP_PATIENCE}")
print("   EMA decay      : " + str(EMA_DECAY))


In [ ]:
# ── Optimizers ────────────────────────────────────────────────────────
opt_G = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, generator.parameters()),
    lr=LR_GENERATOR, weight_decay=0.01, betas=(0.5, 0.999)
)
opt_D = torch.optim.AdamW(
    discriminator.parameters(),
    lr=LR_DISCRIMINATOR, weight_decay=0.01, betas=(0.5, 0.999)

# ── Schedulers ────────────────────────────────────────────────────────
sched_G = torch.optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=EPOCHS, eta_min=1e-6)

# Warmup scheduler for generator
warmup_sched_G = WarmupScheduler(opt_G, warmup_epochs=WARMUP_EPOCHS, base_lr=LR_GENERATOR)
# Main cosine scheduler
sched_G = torch.optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=EPOCHS, eta_min=1e-6)

# Warmup scheduler for discriminator  
warmup_sched_D = WarmupScheduler(opt_D, warmup_epochs=WARMUP_EPOCHS, base_lr=LR_DISCRIMINATOR)
# Main cosine scheduler
sched_D = torch.optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=EPOCHS, eta_min=1e-6)
sched_D = torch.optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=EPOCHS, eta_min=1e-6)
# ── Loss ──────────────────────────────────────────────────────────────
criterion = TryOnCombinedLoss(device=DEVICE)
# ── AMP dtype + separate scalers for G and D ─────────────────────────
# Using two independent GradScalers is required for GANs:
# each optimizer step must be paired with its own scaler.update() call.
dtype   = torch.float16 if PRECISION == 'fp16' else torch.bfloat16
_use_amp = USE_AMP and DEVICE == 'cuda'
scaler_G = torch.amp.GradScaler('cuda') if _use_amp else None
scaler_D = torch.amp.GradScaler('cuda') if _use_amp else None
# ── Checkpoint dir ────────────────────────────────────────────────────
CKPT_DIR = Path("data/arvton/checkpoints/gan")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
# ── Training history ──────────────────────────────────────────────────
history = {"train_G": [], "train_D": [], "val_G": [], "epoch_times": []}
best_val_loss = float('inf')
print("✅ Optimizers, schedulers and losses ready")
print(f"   AMP      : {'ON (' + PRECISION + ')  — two independent GradScalers' if _use_amp else 'OFF'}")
print(f"   Ckpt dir : {CKPT_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  TRAINING LOOP
print("Starting training...")
print("=" * 65)

global_start = time.time()
for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    generator.train()
    discriminator.train()
    g_loss_epoch = 0.0
    d_loss_epoch = 0.0
    n_batches    = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:3d}/{EPOCHS}", leave=False)
    for step, batch in enumerate(pbar):
        person  = batch["person"].to(DEVICE, non_blocking=True)
        garment = batch["garment"].to(DEVICE, non_blocking=True)
        gt      = batch["gt"].to(DEVICE, non_blocking=True)
        inputs = torch.cat([person, garment], dim=1)  # 6-channel input
        # ── Train Discriminator ───────────────────────────────────────
        opt_D.zero_grad()
        with torch.amp.autocast('cuda', dtype=dtype, enabled=_use_amp):
            fake       = generator(inputs).detach()
            real_pred  = discriminator(gt)
            fake_pred  = discriminator(fake)
            # Hinge GAN loss
            loss_D_real = torch.mean(torch.relu(1.0 - real_pred))
            loss_D_fake = torch.mean(torch.relu(1.0 + fake_pred))
            loss_D      = (loss_D_real + loss_D_fake) * 0.5
        if scaler_D:
            scaler_D.scale(loss_D).backward()
            scaler_D.unscale_(opt_D)
            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), GRAD_CLIP_NORM)
            scaler_D.step(opt_D)
            scaler_D.update()          # ← must be called once per optimizer step
        else:
            loss_D.backward()
            opt_D.step()
        # ── Train Generator ───────────────────────────────────────────
        opt_G.zero_grad()
            fake      = generator(inputs)
            pred_fake = discriminator(fake)
            adv_loss  = -torch.mean(pred_fake)
            losses    = criterion(fake, gt)
            loss_G    = losses["total"] + 0.1 * adv_loss  # weighted adversarial
        if scaler_G:
            scaler_G.scale(loss_G).backward()
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                scaler_G.unscale_(opt_G)
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, generator.parameters()),
                    GRAD_CLIP_NORM,
                )
                scaler_G.step(opt_G)
                scaler_G.update()      # ← must match every scaler_G.step()
            loss_G.backward()
                opt_G.step()
                # Update EMA weights
                ema_generator.update()
        g_loss_epoch += loss_G.item()
        d_loss_epoch += loss_D.item()
        n_batches    += 1
        pbar.set_postfix(
            G=f"{loss_G.item():.4f}",
            D=f"{loss_D.item():.4f}",
            lr=f"{opt_G.param_groups[0]['lr']:.1e}",  # safe at every step
        )
    sched_G.step()
    # Apply warmup for first N epochs
    if epoch <= WARMUP_EPOCHS:
        warmup_sched_G.step(epoch)
        warmup_sched_D.step(epoch)
    sched_D.step()
    avg_G = g_loss_epoch / max(n_batches, 1)
    avg_D = d_loss_epoch / max(n_batches, 1)
    history["train_G"].append(avg_G)
    history["train_D"].append(avg_D)
    # ── Validation ────────────────────────────────────────────────────
    generator.eval()
    val_G = 0.0
    with torch.no_grad():
        for batch in val_loader:
            person  = batch["person"].to(DEVICE, non_blocking=True)
            garment = batch["garment"].to(DEVICE, non_blocking=True)
            gt      = batch["gt"].to(DEVICE, non_blocking=True)
            inputs  = torch.cat([person, garment], dim=1)
            with torch.amp.autocast('cuda', dtype=dtype, enabled=_use_amp):
                fake   = generator(inputs)
                losses = criterion(fake, gt)
            val_G += losses["total"].item()
    avg_val = val_G / max(len(val_loader), 1)
    history["val_G"].append(avg_val)
    # ── Early Stopping Check ─────────────────────────────────────────────
    if early_stopping(avg_val):
        print(f"         🛑 Early stopping at epoch {epoch}")
        break
    epoch_secs = time.time() - epoch_start
    history["epoch_times"].append(epoch_secs)
    # ── Print epoch summary ───────────────────────────────────────────
    eta_min = (EPOCHS - epoch) * epoch_secs / 60
    print(
        f"Epoch {epoch:3d}/{EPOCHS}  "
        f"G={avg_G:.4f}  D={avg_D:.4f}  "
        f"Val={avg_val:.4f}  "
        f"[{epoch_secs:.0f}s | ETA {eta_min:.0f}min]"
    )
    # ── VRAM monitoring + cache flush every 10 epochs ─────────────────
    if epoch % 10 == 0 and torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1024**3
        peak = torch.cuda.max_memory_allocated() / 1024**3
        print(f"         VRAM: {used:.2f} GB used / {peak:.2f} GB peak")
        torch.cuda.empty_cache()
    # ── Save checkpoint ───────────────────────────────────────────────
    is_best = avg_val < best_val_loss
    if is_best:
        best_val_loss = avg_val
    if epoch % SAVE_EVERY == 0 or epoch == EPOCHS or is_best:
        tag  = f"epoch_{epoch:03d}" + ("_best" if is_best else "")
        ckpt = CKPT_DIR / f"gan_{tag}.pt"
        torch.save({
            "epoch"        : epoch,
            "generator"    : generator.state_dict(),
            "discriminator": discriminator.state_dict(),
            "opt_G"        : opt_G.state_dict(),
            "opt_D"        : opt_D.state_dict(),
            "scaler_G"     : scaler_G.state_dict() if scaler_G else None,
            "scaler_D"     : scaler_D.state_dict() if scaler_D else None,
            "history"      : history,
            "config"       : {
                "epochs"    : EPOCHS,
                "batch_size": BATCH_SIZE,
                "image_size": IMAGE_SIZE,
                "lora_rank" : LORA_RANK if USE_LORA else None,
                "lr_G"      : LR_GENERATOR,
                "lr_D"      : LR_DISCRIMINATOR,
            },
        }, str(ckpt))
        size_mb = ckpt.stat().st_size / 1024**2
        print(f"         💾 Checkpoint saved: {ckpt.name} ({size_mb:.1f} MB)")
        # Save EMA checkpoint
        if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
            ema_generator.apply_shadow()
            ema_ckpt = CKPT_DIR / f"gan_{tag}_ema.pt"
            torch.save(generator.state_dict(), str(ema_ckpt))
            ema_generator.restore()
            print("         📦 EMA checkpoint saved: " + ema_ckpt.name)
total_min = (time.time() - global_start) / 60
print(f"✅ Training complete in {total_min:.1f} min ({total_min / 60:.2f} hrs)")
print(f"   Best validation loss : {best_val_loss:.4f}")

---
## Step 7 — Plot Loss Curves & Visualize Results

In [ ]:
# ── Loss curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_x = range(1, len(history['train_G']) + 1)
axes[0].plot(epochs_x, history['train_G'], label='Train G', color='#4c8bf5', linewidth=2)
axes[0].plot(epochs_x, history['val_G'],   label='Val G',   color='#ea4335', linewidth=2, linestyle='--')
axes[0].set_title('Generator Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_x, history['train_D'], label='Train D', color='#34a853', linewidth=2)
axes[1].set_title('Discriminator Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plot_path = Path("data/arvton/outputs/training_curves.png")
plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(str(plot_path), dpi=120, bbox_inches='tight')
plt.show()
print(f"✅ Loss curves saved to {plot_path}")

In [ ]:
# ── Visual inference on validation samples ────────────────────────────
from PIL import Image as PILImage
import torchvision.transforms.functional as TF

generator.eval()
NUM_VIZ = min(4, val_size)
fig, axes = plt.subplots(NUM_VIZ, 3, figsize=(12, 4 * NUM_VIZ))
if NUM_VIZ == 1:
    axes = axes.reshape(1, -1)
val_iter  = iter(val_loader)
batch_viz = next(val_iter)
with torch.no_grad():
    persons  = batch_viz["person"][:NUM_VIZ].to(DEVICE)
    garments = batch_viz["garment"][:NUM_VIZ].to(DEVICE)
    gts      = batch_viz["gt"][:NUM_VIZ]
    inputs   = torch.cat([persons, garments], dim=1)
    fakes    = generator(inputs).cpu()
def to_pil(t):
    """Denormalize tensor [-1,1] → PIL image."""
    img = (t.clamp(-1, 1) + 1) / 2
    return TF.to_pil_image(img)
for i in range(NUM_VIZ):
    axes[i, 0].imshow(to_pil(persons[i].cpu()))
    axes[i, 0].set_title(f"Person #{i+1}", fontweight='bold')
    axes[i, 0].axis('off')
    axes[i, 1].imshow(to_pil(garments[i].cpu()))
    axes[i, 1].set_title(f"Garment #{i+1}", fontweight='bold')
    axes[i, 1].axis('off')
    axes[i, 2].imshow(to_pil(fakes[i]))
    axes[i, 2].set_title(f"Try-On Result #{i+1}", color='green', fontweight='bold')
    axes[i, 2].axis('off')
plt.suptitle('ARVTON — Validation Inference Samples', fontsize=16, y=1.02)
plt.tight_layout()
out_path = Path("data/arvton/outputs/val_samples.png")
plt.savefig(str(out_path), dpi=120, bbox_inches='tight')
plt.show()
print(f"✅ Validation samples saved to {out_path}")

---
## Step 8 — Save Best Checkpoint to Google Drive & Download

In [ ]:
# ── Copy all checkpoints to Google Drive ─────────────────────────────
import shutil
from pathlib import Path

local_ckpts = Path("data/arvton/checkpoints/gan")
saved = []
for ckpt in sorted(local_ckpts.glob("*.pt")):
    dest = DRIVE_CKPT / ckpt.name
    shutil.copy2(str(ckpt), str(dest))
    size_mb = dest.stat().st_size / 1024**2
    saved.append((dest, size_mb))
    print(f"  ✅ {ckpt.name:40s} → Drive ({size_mb:.1f} MB)")
# Also save loss curves
for f in ["training_curves.png", "val_samples.png"]:
    src = Path(f"data/arvton/outputs/{f}")
    if src.exists():
        shutil.copy2(str(src), str(DRIVE_LOGS / f))
        print(f"  📊 {f} → Drive")
# Save training history JSON
hist_path = DRIVE_LOGS / "training_history.json"
import json
hist_path.write_text(json.dumps(history, indent=2))
print(f"  📄 training_history.json → Drive")
print(f"\n✅ Saved {len(saved)} checkpoints to Google Drive")

In [ ]:
# ── Download best checkpoint to local machine ─────────────────────────
from google.colab import files

# Find the _best checkpoint (or latest if none)
best_ckpts = sorted(DRIVE_CKPT.glob("*_best.pt"), key=lambda p: p.stat().st_mtime)
all_ckpts  = sorted(DRIVE_CKPT.glob("*.pt"),      key=lambda p: p.stat().st_mtime)
to_download = best_ckpts[-1] if best_ckpts else (all_ckpts[-1] if all_ckpts else None)
if to_download:
    size_mb = to_download.stat().st_size / 1024**2
    print(f"Downloading: {to_download.name} ({size_mb:.1f} MB)")
    print(f"📥 Place this file in your local project at:")
    print(f"   data/arvton/checkpoints/gan/{to_download.name}")
    files.download(str(to_download))
else:
    print("⚠️  No checkpoints found — run training first")

---
## Step 9 — (Optional) Resume Training from Checkpoint

In [ ]:
# ── Resume training from a Drive checkpoint ───────────────────────────
RESUME_EPOCH = 0  # set to the epoch number you want to resume from (0 = skip)

if RESUME_EPOCH > 0:
    pattern = f"*epoch_{RESUME_EPOCH:03d}*.pt"
    ckpts   = list(DRIVE_CKPT.glob(pattern))
    if not ckpts:
        # Fall back to latest available checkpoint
        ckpts = sorted(DRIVE_CKPT.glob("*.pt"), key=lambda p: p.stat().st_mtime)
    if ckpts:
        ckpt_path = sorted(ckpts)[-1]
        state     = torch.load(str(ckpt_path), map_location=DEVICE)
        generator.load_state_dict(state["generator"])
        discriminator.load_state_dict(state["discriminator"])
        opt_G.load_state_dict(state["opt_G"])
        opt_D.load_state_dict(state["opt_D"])
        # Restore AMP scaler states if they were saved
        if scaler_G and state.get("scaler_G"):
            scaler_G.load_state_dict(state["scaler_G"])
        if scaler_D and state.get("scaler_D"):
            scaler_D.load_state_dict(state["scaler_D"])
        history     = state["history"]
        START_EPOCH = state["epoch"] + 1
        print(f"✅ Resumed from: {ckpt_path.name}")
        print(f"   Epoch {state['epoch']} restored — continuing from epoch {START_EPOCH}")
        print(f"   Best val loss so far: {min(history['val_G']):.4f}")
    else:
        print(f"⚠️  No checkpoint found matching epoch {RESUME_EPOCH}")
else:
    print("ℹ️  Resume skipped (RESUME_EPOCH=0) — starting fresh training")

---
## Step 10 — (Optional) Run LoRA Fine-Tuning Script Directly

Alternative to the manual loop above — uses `pipeline/fine_tune.py` directly.

In [ ]:
# ── Run dedicated LoRA fine-tuning pipeline ───────────────────────────
# Uncomment to run (this is an alternative to Step 6)

# !python -m pipeline.train_local \
#     --epochs {EPOCHS} \
#     --batch-size {BATCH_SIZE} \
#     --lr {LR_GENERATOR} \
#     --lora --lora-rank {LORA_RANK} \
#     --amp
print("Uncomment the command above to use the CLI trainer.")

---
## ✅ Summary — What to do after training

| Task | Command |
|------|----|
| 1. On your local machine, copy the `.pt` file | → `data/arvton/checkpoints/gan/` |
| 2. Start backend API | `python run_local.py` |
| 3. Test endpoint | `curl http://localhost:8000/health` |
| 4. Run Flutter app | `cd arvton_app && flutter run -d chrome` |
| 5. Deploy to cloud | See `DEPLOY.md` |
### Project Structure
```
AR-PROTOTYPE-4/
├── pipeline/
│   ├── fine_tune.py       ← LoRA fine-tuning (Phase 6)
│   ├── train_local.py     ← GAN training orchestrator (Phase 8)
│   ├── refine.py          ← RefineGenerator + Discriminator
│   ├── segment.py         ← SAM-2 segmentation
│   ├── tryon.py           ← Leffa / CatVTON try-on
│   ├── reconstruct.py     ← HMR 2.0 + TripoSR 3D
│   └── synthetic_gen.py   ← SDXL synthetic data
├── app/
│   └── main.py            ← FastAPI backend
├── arvton_app/
│   └── lib/               ← Flutter AR mobile app
└── ARVTON_Colab_Training.ipynb  ← This notebook
**Repo:** https://github.com/Gandharv2323/AR-PROTOTYPE-4